# Student Exam Performance: Data Processing App

**Objective:** Build an end-to-end Python pipeline to:
1. Load a raw dataset of student exam records (`StudentsPerformance.csv`)
2. Clean and validate data using custom modular functions
3. Analyze exam scores using NumPy vectorized operations and statistics

*Note: This notebook is compatible with both local Jupyter environments and Google Colab.*

In [ ]:
import csv
import os
import urllib.request
from typing import List, Dict, Any, Optional
import numpy as np

print(f"NumPy version: {np.__version__}")

## 1. Load Raw Dataset
We locate the dataset (with automatic fallback download for Google Colab) and load the raw CSV data into a list of row dictionaries using Python's standard `csv` library.

In [ ]:
# Determine file path (works locally, in src/, or in Google Colab)
DATA_PATH = "../data/StudentsPerformance.csv"
if not os.path.exists(DATA_PATH):
    if os.path.exists("data/StudentsPerformance.csv"):
        DATA_PATH = "data/StudentsPerformance.csv"
    elif os.path.exists("StudentsPerformance.csv"):
        DATA_PATH = "StudentsPerformance.csv"
    else:
        # Automatic fallback for Google Colab
        print("Dataset not found locally; downloading from GitHub repository...")
        url = "https://raw.githubusercontent.com/AbellTT/week1-demo/main/data/StudentsPerformance.csv"
        urllib.request.urlretrieve(url, "StudentsPerformance.csv")
        DATA_PATH = "StudentsPerformance.csv"

print(f"Using dataset: {DATA_PATH}")

def load_raw_data(filepath: str) -> List[Dict[str, str]]:
    with open(filepath, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return list(reader)

raw_data = load_raw_data(DATA_PATH)
print(f"Total raw records loaded: {len(raw_data)}")
print("Sample raw record 1:", raw_data[0])

## 2. Clean the Data Using Modular Functions
Raw datasets often contain dirty formatting, inconsistent casing, and corrupt/missing values.
We define dedicated functions to:
- Safely parse and validate exam scores ($0 \le \text{score} \le 100$)
- Standardize categorical text strings (remove whitespace, normalize casing)
- Map test preparation status to a boolean (`True`/`False`)
- Filter out invalid or corrupted records

In [ ]:
def parse_score(val: Any) -> Optional[int]:
    """Safely converts score to integer between 0 and 100, returning None if invalid."""
    try:
        score = int(float(str(val).strip()))
        if 0 <= score <= 100:
            return score
        return None
    except (ValueError, TypeError):
        return None

def clean_row(row: Dict[str, str]) -> Optional[Dict[str, Any]]:
    """Cleans, normalizes, and validates a single student record."""
    normalized_row = {k.strip().lower(): v.strip() for k, v in row.items()}
    
    math = parse_score(normalized_row.get("math score"))
    reading = parse_score(normalized_row.get("reading score"))
    writing = parse_score(normalized_row.get("writing score"))
    
    if math is None or reading is None or writing is None:
        return None
        
    return {
        "gender": normalized_row.get("gender", "unknown").lower(),
        "race_ethnicity": normalized_row.get("race/ethnicity", "unknown").title(),
        "parental_education": normalized_row.get("parental level of education", "unknown"),
        "lunch": normalized_row.get("lunch", "unknown").lower(),
        "test_prep_completed": normalized_row.get("test preparation course", "").lower() == "completed",
        "math_score": math,
        "reading_score": reading,
        "writing_score": writing,
    }

def clean_dataset(records: List[Dict[str, str]]) -> List[Dict[str, Any]]:
    """Applies cleaning to all records and returns a list of valid records."""
    cleaned = []
    dropped_count = 0
    for row in records:
        cleaned_row = clean_row(row)
        if cleaned_row is not None:
            cleaned.append(cleaned_row)
        else:
            dropped_count += 1
    print(f"Cleaning complete! Valid records: {len(cleaned)}, Dropped records: {dropped_count}")
    return cleaned

In [ ]:
# Execute cleaning pipeline
cleaned_data = clean_dataset(raw_data)

# Preview first cleaned record
print("\nSample cleaned record:")
for key, value in cleaned_data[0].items():
    print(f"  {key}: {value} ({type(value).__name__})")

## 3. Data Processing with NumPy
We convert the cleaned scores into a 2D NumPy array (`shape: (N, 3)`), where columns represent:
- Column 0: **Math**
- Column 1: **Reading**
- Column 2: **Writing**

We use vectorized operations to compute totals, averages, statistics, and boolean filters.

In [ ]:
# Convert cleaned data into 2D NumPy array
scores = np.array(
    [[r["math_score"], r["reading_score"], r["writing_score"]] for r in cleaned_data],
    dtype=np.float64
)

# Vectorized calculations per student (along axis=1)
total_scores = np.sum(scores, axis=1)    # Total score out of 300
avg_scores = np.mean(scores, axis=1)      # Average score out of 100

print(f"Scores Matrix Shape: {scores.shape} (Students, Subjects)")
print(f"First 5 student averages: {np.round(avg_scores[:5], 2)}")
print(f"Highest individual average: {np.max(avg_scores):.2f}")
print(f"Lowest individual average:  {np.min(avg_scores):.2f}")

### 3.1 Descriptive Statistics per Subject
We calculate the mean, median, standard deviation, minimum, maximum, and Interquartile Range (IQR) for each exam subject.

In [ ]:
subjects = ["Math", "Reading", "Writing"]

print(f"{'Subject':<10} | {'Mean':>7} | {'Median':>7} | {'Std Dev':>7} | {'Min':>5} | {'Max':>5} | {'IQR':>7}")
print("-" * 66)

for idx, subj in enumerate(subjects):
    col = scores[:, idx]
    mean_val = np.mean(col)
    median_val = np.median(col)
    std_val = np.std(col)
    min_val = np.min(col)
    max_val = np.max(col)
    iqr_val = np.percentile(col, 75) - np.percentile(col, 25)
    
    print(f"{subj:<10} | {mean_val:>7.2f} | {median_val:>7.2f} | {std_val:>7.2f} | {min_val:>5.0f} | {max_val:>5.0f} | {iqr_val:>7.2f}")

### 3.2 Boolean Masking & Performance Filtering
Using NumPy boolean arrays, we can filter students based on performance thresholds:
1. **Passing Rate:** Students scoring $\ge 60$ in all three subjects.
2. **Top 10% Performers:** Students with average score at or above the 90th percentile.

In [ ]:
# 1. Passing condition: >= 60 in all subjects
passing_mask = np.all(scores >= 60.0, axis=1)
pass_count = int(np.sum(passing_mask))
pass_rate = (pass_count / len(scores)) * 100

# 2. Top 10% cutoff
p90_threshold = float(np.percentile(avg_scores, 90))
top_performers_mask = avg_scores >= p90_threshold
top_count = int(np.sum(top_performers_mask))

print(f"Students passing all subjects (>= 60) : {pass_count}/{len(scores)} ({pass_rate:.1f}%)")
print(f"Top 10% Cutoff Threshold (90th Pct)  : {p90_threshold:.2f} avg score ({top_count} students)")

### 3.3 Comparative Group Analysis: Impact of Test Preparation
We use boolean indexing to compare the average exam performance of students who completed a test preparation course versus those who did not.

In [ ]:
# Extract boolean array for test prep
prep_mask = np.array([r["test_prep_completed"] for r in cleaned_data], dtype=bool)

completed_mean = float(np.mean(avg_scores[prep_mask]))
none_mean = float(np.mean(avg_scores[~prep_mask]))
diff = completed_mean - none_mean

print(f"Completed Test Prep ({int(np.sum(prep_mask))} students): {completed_mean:.2f} average")
print(f"No Test Prep        ({int(np.sum(~prep_mask))} students): {none_mean:.2f} average")
print(f"Performance Difference                    : +{diff:.2f} points higher with preparation course!")

### 3.4 Vectorized Z-Score Normalization
Standardized score formula: 
$$Z = \frac{X - \mu}{\sigma}$$
Z-scores represent how many standard deviations a student's score is from the population mean.

In [ ]:
# Vectorized standardization across columns
means = np.mean(scores, axis=0)
stds = np.std(scores, axis=0)
z_scores = (scores - means) / stds

print("Sample Standardized Z-Scores (First 3 students):")
for i in range(3):
    z_text = ", ".join([f"{subjects[j]}: {z_scores[i, j]:+.2f}" for j in range(3)])
    print(f"  Student {i+1} -> {z_text}")

## 4. Key Findings & Conclusion
- **Overall Performance:** Reading has the highest overall average score (69.17), while Math has the lowest (66.09).
- **Pass Rate:** 60.3% of students achieved a passing grade ($\ge 60$) in all three subjects.
- **Test Preparation Course:** Students who completed the test preparation course scored on average **7.63 points higher** (72.67 vs 65.04) than those who did not, indicating a clear positive correlation.